# Pilot Experiment for Hypothesis H2.1 on VLM (COCO val2014, 20 images)

## 0. Goal & Hypotheses
- **(P0, premise):** Visual contribution ($V_t$) decreases with position in the generated sequence.
- **(P1):** The probability that a generated object changes when a preceding token is perturbed ($S_t$, flip rate) increases with the object's position ($t$).
- **(P2):** At the same position, objects with low visual contribution change more easily (negative correlation between $S_t$ and $V_t$).

> **Note:** All findings are labeled **exploratory ($n=20$ images)**. Pipeline optimized for Kaggle 2 x NVIDIA T4 (16 GB each).

### Step 0: Clone Codebase từ GitHub
Nếu bạn chỉ tải file notebook lên Kaggle, ô này sẽ tự động tải các file code (`run_pilot.py`, `config.py`, ...) từ GitHub về môi trường Kaggle.

In [ ]:
# === BUOC 1: TAI CODEBASE TU GITHUB NEU CHUA CO ===
import os
if not os.path.exists("run_pilot.py"):
    if not os.path.exists("/kaggle/working/Experiment_CTI"):
        !git clone https://github.com/ntmy12/Experiment_CTI.git /kaggle/working/Experiment_CTI
    %cd /kaggle/working/Experiment_CTI
print("Working directory:", os.getcwd())
!ls -la run_pilot.py config.py


### Step 1: Environment & Dependency Installation

In [ ]:
# Cai dat cac goi phu thuoc can thiet (khong dung --upgrade de tranh conflict voi cudf/pandas cua Kaggle)
!pip install -q transformers accelerate pycocotools statsmodels


### Step 2: GPU and Hardware Verification

In [ ]:
import torch, os
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU Device Count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB)")

### Step 3: Run Full End-to-End Pilot Pipeline (20 Images)
Runs greedy generation, teacher-forcing evaluation, prefix perturbations, robustness checks, statistical bootstrap analyses, and visualizations with live unbuffered logging and checkpointing.

In [ ]:
# === CAU HINH SO LUONG ANH TAI DAY ===
N_IMAGES = 20  # Thay doi so anh ban muon chay tai day (vi du: 5, 20, 50)

import os
os.environ["PYTHONUNBUFFERED"] = "1"
!python -u run_pilot.py --n_images {N_IMAGES}


### Step 4: Inspect Generated Artifacts

In [ ]:
import os
results_dir = "/kaggle/working/results" if os.path.exists("/kaggle/working/results") else "./results"
for root, dirs, files in os.walk(results_dir):
    level = root.replace(results_dir, '').count(os.sep)
    indent = ' ' * 4 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        fpath = os.path.join(root, f)
        size_kb = os.path.getsize(fpath) / 1024.0
        print(f"{subindent}{f} ({size_kb:.1f} KB)")

### Step 5: Display Standalone HTML Report Directly in Notebook

In [ ]:
from IPython.display import HTML, display
import os

report_path = "/kaggle/working/results/report.html" if os.path.exists("/kaggle/working/results/report.html") else "./results/report.html"
if os.path.exists(report_path):
    with open(report_path, "r", encoding="utf-8") as f:
        html_content = f.read()
    display(HTML(html_content))
else:
    print(f"Report not found at {report_path}")

### Step 6: Export Results Archive for Download

In [ ]:
import shutil, os
src = "/kaggle/working/results" if os.path.exists("/kaggle/working/results") else "./results"
dst = "/kaggle/working/pilot_h21_results" if os.path.exists("/kaggle") else "./pilot_h21_results"
shutil.make_archive(dst, 'zip', src)
print(f"Created archive: {dst}.zip ({os.path.getsize(dst + '.zip') / 1024 / 1024:.2f} MB)")